In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Milestone 5

In [6]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

prompt = train_df.loc[25, "prompt"]

deberta_model_path = "microsoft/deberta-v3-small"
roberta_model_path = "roberta-base"

deberta_tokenizer = AutoTokenizer.from_pretrained(deberta_model_path)
deberta_model = AutoModelForSequenceClassification.from_pretrained(
    deberta_model_path,
    num_labels=5
)

roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_model_path)
roberta_model = AutoModelForSequenceClassification.from_pretrained(
    roberta_model_path,
    num_labels=5
)

deberta_model.eval()
roberta_model.eval()

inputs = deberta_tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=512
)

with torch.no_grad():
    outputs = deberta_model(**inputs)

probabilities = torch.softmax(outputs.logits, dim=1)

label_map = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E"
}

predicted_id = probabilities.argmax(dim=1).item()
predicted_option = label_map[predicted_id]
predicted_probability = probabilities[0][predicted_id].item()

print(f"{predicted_option}, {predicted_probability:.6f}")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight     

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


A, 0.342773


In [7]:
deberta_inputs = deberta_tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=512
)

roberta_inputs = roberta_tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=512
)

with torch.no_grad():
    deberta_probs = torch.softmax(deberta_model(**deberta_inputs).logits, dim=1)
    roberta_probs = torch.softmax(roberta_model(**roberta_inputs).logits, dim=1)

avg_probs = (deberta_probs + roberta_probs) / 2

label_map = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E"
}

pred = avg_probs.argmax(dim=1).item()

print(f"{label_map[pred]}, {avg_probs[0][pred].item():.6f}")

A, 0.270675


In [8]:
deberta_inputs = deberta_tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=512
)

roberta_inputs = roberta_tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=512
)

with torch.no_grad():
    deberta_probs = torch.softmax(deberta_model(**deberta_inputs).logits, dim=1)
    roberta_probs = torch.softmax(roberta_model(**roberta_inputs).logits, dim=1)

weighted_probs = 0.7 * deberta_probs + 0.3 * roberta_probs

label_map = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E"
}

pred = weighted_probs.argmax(dim=1).item()

print(f"{label_map[pred]}, {weighted_probs[0][pred].item():.6f}")

A, 0.299563


In [9]:
top3 = torch.topk(weighted_probs, k=3, dim=1).indices[0].tolist()

label_map = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E"
}

print(" ".join(label_map[i] for i in top3))

A D B


In [10]:
print(len(test_df))

500


In [11]:
different = 0

for i in range(50):
    prompt = test_df.loc[i, "prompt"]
    augmented_prompt = "Answer the following multiple-choice question carefully: " + prompt

    inputs1 = deberta_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    inputs2 = deberta_tokenizer(
        augmented_prompt,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    with torch.no_grad():
        probs1 = torch.softmax(deberta_model(**inputs1).logits, dim=1)
        probs2 = torch.softmax(deberta_model(**inputs2).logits, dim=1)

    pred1 = probs1.argmax(dim=1).item()
    pred_tta = ((probs1 + probs2) / 2).argmax(dim=1).item()

    if pred1 != pred_tta:
        different += 1

print(different)

2


In [12]:
different = 0

for i in range(100):
    prompt = test_df.loc[i, "prompt"]

    deberta_inputs = deberta_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    roberta_inputs = roberta_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    with torch.no_grad():
        deberta_probs = torch.softmax(deberta_model(**deberta_inputs).logits, dim=1)
        roberta_probs = torch.softmax(roberta_model(**roberta_inputs).logits, dim=1)

    weighted_probs = 0.7 * deberta_probs + 0.3 * roberta_probs

    deberta_pred = deberta_probs.argmax(dim=1).item()
    ensemble_pred = weighted_probs.argmax(dim=1).item()

    if deberta_pred != ensemble_pred:
        different += 1

print(different)

0


In [13]:
positive_gain = 0

for i in range(100):
    prompt = test_df.loc[i, "prompt"]

    deberta_inputs = deberta_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    roberta_inputs = roberta_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    with torch.no_grad():
        deberta_probs = torch.softmax(deberta_model(**deberta_inputs).logits, dim=1)
        roberta_probs = torch.softmax(roberta_model(**roberta_inputs).logits, dim=1)

    weighted_probs = 0.7 * deberta_probs + 0.3 * roberta_probs

    deberta_conf = deberta_probs.max().item()
    ensemble_conf = weighted_probs.max().item()

    if ensemble_conf > deberta_conf:
        positive_gain += 1

print(positive_gain)

0


In [14]:
changed = 0

for i in range(100):
    prompt = test_df.loc[i, "prompt"]

    deberta_inputs = deberta_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    roberta_inputs = roberta_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    with torch.no_grad():
        deberta_probs = torch.softmax(deberta_model(**deberta_inputs).logits, dim=1)
        roberta_probs = torch.softmax(roberta_model(**roberta_inputs).logits, dim=1)

    weighted_probs = 0.7 * deberta_probs + 0.3 * roberta_probs

    deberta_top3 = torch.topk(deberta_probs, k=3, dim=1).indices[0].tolist()
    ensemble_top3 = torch.topk(weighted_probs, k=3, dim=1).indices[0].tolist()

    if deberta_top3 != ensemble_top3:
        changed += 1

print(changed)

37


In [21]:
from sklearn.model_selection import train_test_split
import torch


_, valid_df = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)
valid_df = valid_df.reset_index(drop=True)

label_map = {0: "A", 1: "B", 2: "C", 3: "D", 4: "E"}

def apk(actual, predicted, k=3):
    for rank, pred in enumerate(predicted[:k], start=1):
        if pred == actual:
            return 1.0 / rank
    return 0.0

scores = []

for i in range(100):
    row = valid_df.iloc[i]
    prompt = row["prompt"]
    actual = row["answer"]

    deberta_inputs = deberta_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    roberta_inputs = roberta_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    with torch.no_grad():
        deberta_probs = torch.softmax(deberta_model(**deberta_inputs).logits, dim=1)
        roberta_probs = torch.softmax(roberta_model(**roberta_inputs).logits, dim=1)

    weighted_probs = 0.7 * deberta_probs + 0.3 * roberta_probs

    top3 = torch.topk(weighted_probs, k=3).indices[0].tolist()
    preds = [label_map[j] for j in top3]

    scores.append(apk(actual, preds))

map3 = sum(scores) / len(scores)
print(f"{map3:.4f}")

0.3533
